In [ ]:
import mlflow
import pandas as pd
import plotly.express as px
import dotenv
import os
from mlflow import MlflowClient
from typing import Optional
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from tqdm import tqdm

# Load environment variables from .env file
dotenv.load_dotenv()
mlflow_uri = os.getenv("MLFLOW_TRACKING_URI")

# Set tracking URI (adjust if remote)
mlflow.set_tracking_uri(mlflow_uri)

# Choose experiment by name or ID
experiment_name = "STASC"
experiment = mlflow.get_experiment_by_name(experiment_name)

dr = mlflow.search_runs(experiment_ids=[experiment.experiment_id])


methods = ["cove", "baseline_cot", "baseline_no_cot", "stasc_FNF", "stasc_FNE", "stasc_FIF", "stasc_FIE", "stasc_ENF", "stasc_ENE", "stasc_EIF", "stasc_EIE"] 
datasets = ["nq"]
datasets = ["hotpot"]

In [10]:
def collect_metrics(client: MlflowClient, run_id: str, run_name: Optional[str] = None) -> pd.DataFrame:

    keys = ["grad_norm", "learning_rate", "epoch", "loss"]
    df_final = pd.DataFrame()
    for key in keys:
        metrics = client.get_metric_history(run_id, key)
        tmp = pd.DataFrame([dict(metric) for metric in metrics]).sort_values("step")
        tmp = tmp.rename(columns={"value": key})
        df_final = (
            pd.merge(df_final, tmp[["step", key]], on="step", how="outer")
            if not df_final.empty
            else tmp[["step", key]]
        )

    df_final = df_final.sort_values("step").reset_index(drop=True)
    df_final.insert(0, "run_id", run_id)
    if run_name is None:
        run = client.get_run(run_id)
        run_name = run.data.tags.get("mlflow.runName", "unknown")
    df_final.insert(1, "run_name", run_name)
    return df_final



def plot_loss(df: pd.DataFrame, y_axis: str = "loss") -> None:
    methods = df['method'].unique()
    fig = make_subplots(rows=2, cols=2, shared_yaxes=True, subplot_titles=methods)
    window = [(1,1), (1,2), (2,1), (2,2)]
    for (row, col), method in zip(window, methods):

        for run_name in df['run_name'].unique():
            df_run = df[df['method'] == method]
            df_run = df_run[df_run['run_name'] == run_name]
            fig.add_trace(
                go.Scatter(x=df_run['global_step'], y=df_run[y_axis], mode='lines', name=run_name,  legendgroup=f"group_{run_name}"),
                row=row, col=col
            )

    fig.update_layout(height=600, width=1200, title_text=f"{y_axis} over Steps by Method", showlegend=True)
    fig.show()

In [11]:

client = MlflowClient(mlflow_uri)

df_final = pd.DataFrame()
for method in tqdm(methods[3:], desc="Experiments"):
    run_id = mlflow.search_runs(experiment_ids=[experiment.experiment_id], filter_string=f"run_name LIKE '{method}_%' and params.dataset_name = '{datasets[0]}'", order_by=["start_time DESC"])["run_id"][0]
    child_runs = dr[dr["tags.mlflow.parentRunId"] == run_id]
    fine_tune_runs = child_runs[child_runs["tags.mlflow.runName"].str.contains("fine-tune", na=False)].sort_values("tags.mlflow.runName")

    tmp = pd.DataFrame()
    for run_id in tqdm(fine_tune_runs["run_id"], desc=f"{method} fine-tune runs"):
        tmp = pd.concat([tmp, collect_metrics(client, run_id)], ignore_index=True)
    tmp.insert(0, "method", method)
    df_final = pd.concat([df_final, tmp], ignore_index=True)


Experiments: 100%|██████████| 8/8 [00:19<00:00,  2.38s/it]


In [12]:
df_non_decreasing = df_final.sort_values(["run_name", "step"]).reset_index(drop=True)
df_non_decreasing = df_non_decreasing[df_non_decreasing["method"].str.contains(r"_[EF]N", regex=True)]
df_non_decreasing["global_step"] = range(1, len(df_non_decreasing) + 1)

df_improving = df_final.sort_values(["run_name", "step"]).reset_index(drop=True)
df_improving = df_improving[df_improving["method"].str.contains(r"_[EF]I", regex=True)]
df_improving["global_step"] = range(1, len(df_improving) + 1)


plot_loss(df_non_decreasing, "loss")
plot_loss(df_improving, "loss")